# Final Project Experiments

## Setup

In [2]:
import cv2 as cv
import os
import numpy as np
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import csv

INPUT_FILEPATH = "data/input/"
OUTPUT_FILEPATH = "data/output/"
PREDICTION_FILEPATH = "data/predicted/"
HISTEQ_FILEPATH = "data/histeq/"

RAND_SEED = 6524
np.random.seed(RAND_SEED)

NOISE_GAMMA = 3.0                   # Fixed gamma for noise experiments
NOISE_SIGMAS = [0.025, 0.05, 0.1]   # Sigma values for gaussian noise

DARKEN_GAMMAS = [3.0, 4.0, 5.0]     # Gamma values for synthetic darkening

## Helper Functions and Experiments

In [3]:
def add_noise(image, sigma):
    # Rescale image to [0, 1]
    rescaled_image = image / 255

    # Generate Gaussian noise
    noise = np.random.normal(0, sigma, (image.shape[0], image.shape[1], image.shape[2]))

    noisy_image = rescaled_image + noise                    # Add noise
    noisy_image = noisy_image * 255.0                       # Scale image back to [0, 255]
    noisy_image = np.clip(noisy_image, 0, 255)              # Clip to range [0, 255]
    noisy_image = np.round(noisy_image).astype(np.uint8)    # Convert to integers
    
    return noisy_image

def gamma_adjust(image, gamma):

    # Convert image to HSV
    hsv_image = cv.cvtColor(image, cv.COLOR_BGR2HSV)
    
    # Scale intensity values to [0, 1]
    scaled_intensity = hsv_image[:, :, 2] / 255

    gamma_intensity = np.power(scaled_intensity, gamma)           # Apply gamma correction

    gamma_intensity = gamma_intensity * 255.0                       # Scale image back to [0, 255]
    gamma_intensity = np.clip(gamma_intensity, 0, 255)              # Clip to range [0, 255]
    gamma_intensity = np.round(gamma_intensity).astype(np.uint8)    # Convert to integers

    hsv_image[:, :, 2] = gamma_intensity

    output = cv.cvtColor(hsv_image, cv.COLOR_HSV2BGR)

    return output

def apply_histeq(image):
    hsv_image = cv.cvtColor(image, cv.COLOR_BGR2HSV)

    new_V = cv.equalizeHist(hsv_image[:, :, 2])

    hsv_image[:, :, 2] = new_V

    output = cv.cvtColor(hsv_image, cv.COLOR_HSV2BGR)

    return output

def noise_experiments(image, name, ext):

    for sigma in NOISE_SIGMAS:
        
        gamma_image = gamma_adjust(image, NOISE_GAMMA)  # Apply gamma correction to darken image
        noisy_image = add_noise(gamma_image, sigma)     # Add noise
        histeq_image = apply_histeq(noisy_image)

        # Ouput with corresponding name
        output_name = "{}$noise_sigma_{}.{}".format(name, str(sigma).replace(".", "-"), ext)
        cv.imwrite(OUTPUT_FILEPATH + output_name, noisy_image)
        cv.imwrite(HISTEQ_FILEPATH + output_name, histeq_image)

# Gamma values to be tested
def gamma_experiments(image, name, ext):

    for gam in DARKEN_GAMMAS:
        gamma_image = gamma_adjust(image, gam)  # Apply gamma correction to darken image
        histeq_image = apply_histeq(gamma_image)

        # Ouput with corresponding name
        output_name = "{}$gamma_{}.{}".format(name, str(gam).replace(".", "-"), ext)
        cv.imwrite(OUTPUT_FILEPATH + output_name, gamma_image)
        cv.imwrite(HISTEQ_FILEPATH + output_name, histeq_image)


## Running Experiments

In [4]:
# Remove any existing files
existing_files = os.listdir(OUTPUT_FILEPATH)
existing_files.remove(".gitignore") # Don't remove .gitignore

for file in existing_files:
    os.remove(OUTPUT_FILEPATH + file)

# Processing images
input_filenames = os.listdir(INPUT_FILEPATH)
input_filenames.remove(".gitignore") # Don't try to process .gitignore as an image

for name_file in input_filenames:

    name, ext = name_file.split(".")

    image = cv.imread(INPUT_FILEPATH + name_file)

    noise_experiments(image, name, ext)
    gamma_experiments(image, name, ext)

## Evaluate Predicted Outputs

### Create dictionaries to hold data

In [5]:
noise_exp_keys = list()
gamma_exp_keys = list()

experiment_dict = dict()
exp_metrics = dict()

histeq_dict = dict()
histeq_metrics = dict()

for sigma in NOISE_SIGMAS:
    key = "noise_sigma_{}".format(str(sigma).replace(".", "-"))
    noise_exp_keys.append(key)

    experiment_dict[key] = dict()
    exp_metrics[key] = dict()

    histeq_dict[key] = dict()
    histeq_metrics[key] = dict()

for gam in DARKEN_GAMMAS:
    key = "gamma_{}".format(str(gam).replace(".", "-"))
    gamma_exp_keys.append(key)
    
    experiment_dict[key] = dict()
    exp_metrics[key] = dict()

    histeq_dict[key] = dict()
    histeq_metrics[key] = dict()


### Calculate metrics

In [6]:
input_filenames = os.listdir(INPUT_FILEPATH)
input_filenames.remove(".gitignore")    # Skip .gitignore

# Load input images into memory
original_images = dict()

for name_file in input_filenames:
    name, ext = name_file.split(".")

    in_image = cv.imread(INPUT_FILEPATH + name_file)

    original_images[name] = in_image

# Load predicted images into memory
pred_filenames = os.listdir(PREDICTION_FILEPATH)
pred_filenames.remove(".gitignore")     # Skip .gitignore

for name_file in pred_filenames:
    name, ext = name_file.split(".")
    original_name, exp_key = name.split("$")

    pred_image = cv.imread(PREDICTION_FILEPATH + name_file)
    
    experiment_dict[exp_key][original_name] = pred_image

# Load histogram equalized images into memory
histeq_filenames = os.listdir(HISTEQ_FILEPATH)
histeq_filenames.remove(".gitignore")  # Skip .gitignore

for name_file in histeq_filenames:
    name, ext = name_file.split(".")
    original_name, exp_key = name.split("$")

    histeq_image = cv.imread(HISTEQ_FILEPATH + name_file)

    histeq_dict[exp_key][original_name] = histeq_image

In [7]:
""" Zero-DCE """
# Traverse through each experiment
for exp_key, pred_images in experiment_dict.items():

    # Create a dictionary for each metric
    exp_metrics[exp_key]["PSNR"] = dict()
    exp_metrics[exp_key]["SSIM"] = dict()

    # Traverse through all of the images
    for image_name, original_image in original_images.items():
        
        # Retrieve predicted image
        pred_image = pred_images[image_name]

        # Save metrics
        exp_metrics[exp_key]["PSNR"][image_name] = peak_signal_noise_ratio(original_image, pred_image)
        exp_metrics[exp_key]["SSIM"][image_name] = structural_similarity(original_image, pred_image, channel_axis=2)

""" Histogram Equalization """
# Traverse through each experiment
for exp_key, histeq_images in histeq_dict.items():

    # Create a dictionary for each metric
    histeq_metrics[exp_key]["PSNR"] = dict()
    histeq_metrics[exp_key]["SSIM"] = dict()

    # Traverse through all of the images
    for image_name, original_image in original_images.items():
        
        # Retrieve predicted image
        histeq_image = histeq_images[image_name]

        # Save metrics
        histeq_metrics[exp_key]["PSNR"][image_name] = peak_signal_noise_ratio(original_image, histeq_image)
        histeq_metrics[exp_key]["SSIM"][image_name] = structural_similarity(original_image, histeq_image, channel_axis=2)


### Output data CSV

In [8]:
output_csv = open("output_metrics.csv", "w", newline="")

writer = csv.writer(output_csv)

header = ["", "Mean PSNR", "Median PSNR", "Mean SSIM", "Median SSIM"]
writer.writerow(header)

for exp_key, metrics in exp_metrics.items():
    metric_list = ["(Zero-DCE) {}".format(exp_key)]

    for metric_name, metric_values in metrics.items():
        
        values = list(metric_values.values())
        mean_value = np.mean(values).round(5)
        median_value = np.median(values).round(5)

        metric_list.append(mean_value)
        metric_list.append(median_value)

    writer.writerow(metric_list)

for exp_key, metrics in histeq_metrics.items():
    metric_list = ["(Hist. Eq.) {}".format(exp_key)]

    for metric_name, metric_values in metrics.items():
        
        values = list(metric_values.values())
        mean_value = np.mean(values).round(5)
        median_value = np.median(values).round(5)

        metric_list.append(mean_value)
        metric_list.append(median_value)

    writer.writerow(metric_list)

output_csv.close()


c:\Users\Ben\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Ben\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
